# Task 4

In [3]:
from google.colab import files
uploaded = files.upload()

Saving archive (4).zip to archive (4).zip


In [4]:
import zipfile
import os

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("loan_data")

print("Files extracted:")
print(os.listdir("loan_data"))

Files extracted:
['test_Y3wMUE5_7gLdaTN.csv', 'train_u6lujuX_CVtuZ9i.csv']


 1. Understanding Data – فهم البيانات

يهدف هذا المشروع إلى تحليل بيانات طلبات القروض البنكية، وفهم العوامل التي قد تؤثر على قرار الموافقة أو الرفض.

- **المشكلة:** يريد البنك معرفة العوامل المرتبطة بالموافقة على طلب القرض.
- **Target:** العمود Loan_Status، ويمثل نتيجة الطلب (1 للموافقة و0 للرفض).
- **Features:** بقية الأعمدة التي تصف مقدم الطلب وبيانات القرض، مثل الدخل والتاريخ الائتماني والموقع وحالة التعليم.

In [5]:
import pandas as pd

df = pd.read_csv("loan_data/train_u6lujuX_CVtuZ9i.csv")

print("حجم البيانات:")
print(df.shape)

print("\nأول 5 صفوف:")
display(df.head())

print("\nأنواع الأعمدة:")
print(df.dtypes)

print("\nمعلومات البيانات:")
df.info()

حجم البيانات:
(614, 13)

أول 5 صفوف:


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y



أنواع الأعمدة:
Loan_ID               object
Gender                object
Married               object
Dependents            object
Education             object
Self_Employed         object
ApplicantIncome        int64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area         object
Loan_Status           object
dtype: object

معلومات البيانات:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null 

 .1 فحص القيم الناقصة

أفحص القيم الناقصة في الأعمدة لمعرفة البيانات التي تحتاج إلى تنظيف ومعالجة قبل التحليل.

In [6]:
print("Missing values:")
print(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())

Missing values:
Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

Total missing values:
149


  فحص الصفوف المكررة

أتحقق من وجود صفوف مكررة، لأن التكرار قد يؤثر على دقة التحليل والنتائج.

In [7]:
print("Number of duplicated rows:")
print(df.duplicated().sum())

Number of duplicated rows:
0


  الإحصائيات الوصفية

أستخدم الإحصائيات الوصفية لفهم توزيع القيم الرقمية، مثل الدخل ومبلغ القرض ومدة السداد.

In [8]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


  أهم ملاحظات استكشاف البيانات

1. تحتوي البيانات على 614 صفًا و13 عمودًا.
2. يوجد 149 قيمة ناقصة موزعة على عدة أعمدة.
3. لا توجد صفوف مكررة.
4. متوسط دخل مقدم الطلب 5403.46.
5. متوسط دخل الشريك 1621.25.
6. متوسط مبلغ القرض 146.41 ألف.
7. أعلى دخل لمقدم الطلب 81000، وهو أعلى بكثير من المتوسط.
8. عمود Credit_History يحتوي على 50 قيمة ناقصة، لذلك يحتاج إلى معالجة خاصة.

. Cleaning + Decision Log – تنظيف البيانات

اتخذت قرارات التنظيف التالية بناءً على نوع البيانات وطبيعة كل عمود:

| العمود | القرار | السبب |
|---|---|---|
| Gender | تعويض القيم الناقصة بالمنوال | لأنه عمود تصنيفي |
| Married | التعويض بالمنوال | لأنه عمود تصنيفي |
| Dependents | التعويض بالمنوال | للحفاظ على أكثر فئة تكرارًا |
| Self_Employed | التعويض بالمنوال | لأنه عمود تصنيفي |
| LoanAmount | التعويض بالوسيط | لتقليل تأثير القيم المرتفعة |
| Loan_Amount_Term | التعويض بالوسيط | لأنه عمود رقمي |
| Credit_History | تعويض القيم الناقصة بـ -1 | لتمييز القيم المفقودة عن 0 و1 |
| Dependents | تحويل 3+ إلى 3 | لتسهيل التعامل مع القيم |
| Loan_Status | تحويل Y إلى 1 وN إلى 0 | لتجهيز الهدف للتحليل والنمذجة |

In [9]:
df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])
df["Married"] = df["Married"].fillna(df["Married"].mode()[0])
df["Dependents"] = df["Dependents"].fillna(df["Dependents"].mode()[0])
df["Self_Employed"] = df["Self_Employed"].fillna(df["Self_Employed"].mode()[0])

df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["Loan_Amount_Term"] = df["Loan_Amount_Term"].fillna(df["Loan_Amount_Term"].median())

df["Credit_History"] = df["Credit_History"].fillna(-1)

df["Dependents"] = df["Dependents"].replace("3+", "3")
df["Loan_Status"] = df["Loan_Status"].map({"Y": 1, "N": 0})

print("Missing values after cleaning:")
print(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())

Missing values after cleaning:
Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

Total missing values:
0


In [10]:
df.to_csv("loan_clean.csv", index=False)

print("File saved successfully: loan_clean.csv")
print("Shape:", df.shape)

File saved successfully: loan_clean.csv
Shape: (614, 13)


 أثر قرارات التنظيف

بعد تطبيق قرارات التنظيف:
- انخفض عدد القيم الناقصة من 149 إلى 0.
- بقي عدد الصفوف 614 دون حذف أي سجل.
- بقي عدد الأعمدة 13.
- تم تحويل Loan_Status إلى قيم رقمية لتسهيل التحليل.
- تم توحيد قيمة Dependents من 3+ إلى 3.
- استخدمت المنوال للأعمدة التصنيفية والوسيط للأعمدة الرقمية لتقليل تأثير القيم المتطرفة.
- احتفظت بالقيم الناقصة في Credit_History على شكل -1 حتى أميزها عن القيم الأصلية.

 3. Feature Engineering – هندسة الخصائص

أنشأت خصائص جديدة تساعد على فهم الوضع المالي لمقدم الطلب بشكل أفضل:

1. Total_Income: مجموع دخل مقدم الطلب ودخل الشريك.
2. Has_Coapplicant: يوضح وجود شريك في الطلب من عدمه.
3. Loan_to_Income: نسبة مبلغ القرض إلى إجمالي الدخل.
4. Monthly_Payment: تقدير القسط الشهري دون احتساب الفوائد.
5. Payment_to_Income: نسبة القسط الشهري إلى إجمالي الدخل.
6. Payment_Group: تصنيف نسبة القسط إلى أربع فئات.
7. Income_Group: تصنيف دخل مقدم الطلب إلى أربع فئات.

أنشأت هذه الخصائص لتسهيل مقارنة الوضع المالي بين المتقدمين وفهم علاقته بقرار الموافقة.

In [11]:
df["Total_Income"] = df["ApplicantIncome"] + df["CoapplicantIncome"]

df["Has_Coapplicant"] = (df["CoapplicantIncome"] > 0).astype(int)

df["Loan_to_Income"] = (df["LoanAmount"] * 1000) / df["Total_Income"]

df["Monthly_Payment"] = (df["LoanAmount"] * 1000) / df["Loan_Amount_Term"]

df["Payment_to_Income"] = df["Monthly_Payment"] / df["Total_Income"]

labels = ["Low", "Medium-Low", "Medium-High", "High"]

df["Payment_Group"] = pd.qcut(df["Payment_to_Income"], 4, labels=labels)

df["Income_Group"] = pd.qcut(df["ApplicantIncome"], 4, labels=labels)

print("New features:")
display(df[[
    "Total_Income",
    "Has_Coapplicant",
    "Loan_to_Income",
    "Monthly_Payment",
    "Payment_to_Income",
    "Payment_Group",
    "Income_Group"
]].head())

New features:


,Total_Income,Has_Coapplicant,Loan_to_Income,Monthly_Payment,Payment_to_Income,Payment_Group,Income_Group
0,5849.0,0,21.884083,355.555556,0.060789,Medium-Low,High
1,6091.0,1,21.014612,355.555556,0.058374,Medium-Low,Medium-High
2,3000.0,0,22.000000,183.333333,0.061111,Medium-Low,Medium-Low
3,4941.0,1,24.286582,333.333333,0.067463,Medium-Low,Low
4,6000.0,0,23.500000,391.666667,0.065278,Medium-Low,High


In [12]:
df.to_csv("loan_features.csv", index=False)

print("File saved successfully: loan_features.csv")
print("Shape:", df.shape)

File saved successfully: loan_features.csv
Shape: (614, 20)


. Feature Selection – اختيار الخصائص

في هذه المرحلة أحدد الخصائص المفيدة للتنبؤ بقرار الموافقة على القرض، وأستبعد الأعمدة التعريفية أو الخصائص التي لا تضيف معلومات كافية.

In [13]:
print("Unique values in each column:")
print(df.nunique())

Unique values in each column:
Loan_ID              614
Gender                 2
Married                2
Dependents             4
Education              2
Self_Employed          2
ApplicantIncome      505
CoapplicantIncome    287
LoanAmount           203
Loan_Amount_Term      10
Credit_History         3
Property_Area          3
Loan_Status            2
Total_Income         554
Has_Coapplicant        2
Loan_to_Income       608
Monthly_Payment      237
Payment_to_Income    611
Payment_Group          4
Income_Group           4
dtype: int64


In [14]:
categorical_cols = [
    "Gender", "Married", "Dependents",
    "Education", "Self_Employed",
    "Credit_History", "Property_Area",
    "Has_Coapplicant", "Payment_Group",
    "Income_Group"
]

for col in categorical_cols:
    print("\nApproval rate by", col)
    print(df.groupby(col, observed=False)["Loan_Status"].mean() * 100)


Approval rate by Gender
Gender
Female    66.964286
Male      69.123506
Name: Loan_Status, dtype: float64

Approval rate by Married
Married
No     62.910798
Yes    71.820449
Name: Loan_Status, dtype: float64

Approval rate by Dependents
Dependents
0    68.611111
1    64.705882
2    75.247525
3    64.705882
Name: Loan_Status, dtype: float64

Approval rate by Education
Education
Graduate        70.833333
Not Graduate    61.194030
Name: Loan_Status, dtype: float64

Approval rate by Self_Employed
Self_Employed
No     68.796992
Yes    68.292683
Name: Loan_Status, dtype: float64

Approval rate by Credit_History
Credit_History
-1.0    74.000000
 0.0     7.865169
 1.0    79.578947
Name: Loan_Status, dtype: float64

Approval rate by Property_Area
Property_Area
Rural        61.452514
Semiurban    76.824034
Urban        65.841584
Name: Loan_Status, dtype: float64

Approval rate by Has_Coapplicant
Has_Coapplicant
0    64.835165
1    71.847507
Name: Loan_Status, dtype: float64

Approval rate by Pay

In [15]:
approval_gaps = {}

for col in categorical_cols:
    rates = df.groupby(col, observed=False)["Loan_Status"].mean() * 100
    approval_gaps[col] = rates.max() - rates.min()

gap_df = pd.DataFrame(
    list(approval_gaps.items()),
    columns=["Feature", "Approval_Gap"]
).sort_values("Approval_Gap", ascending=False)

display(gap_df)

,Feature,Approval_Gap
5,Credit_History,71.713779
6,Property_Area,15.371520
2,Dependents,10.541642
3,Education,9.639303
8,Payment_Group,9.570495
1,Married,8.909651
7,Has_Coapplicant,7.012342
0,Gender,2.159220
9,Income_Group,1.099228
4,Self_Employed,0.504310


In [16]:
numeric_cols = df.select_dtypes(include=["number"]).columns

correlations = df[numeric_cols].corr()["Loan_Status"].sort_values(
    ascending=False
)

print("Correlation with Loan_Status:")
display(correlations)

Correlation with Loan_Status:


,Loan_Status
Loan_Status,1.000000
Credit_History,0.280181
Has_Coapplicant,0.075165
ApplicantIncome,-0.004710
Monthly_Payment,-0.011757
Loan_Amount_Term,-0.022549
Total_Income,-0.031271
LoanAmount,-0.033214
Payment_to_Income,-0.051361
CoapplicantIncome,-0.059187


In [17]:
print("Correlation between ApplicantIncome and Total_Income:")
print(df["ApplicantIncome"].corr(df["Total_Income"]))

Correlation between ApplicantIncome and Total_Income:
0.8930372940684421


  جدول قرارات اختيار الخصائص

| Feature | القرار | الدليل والسبب |
|---|---|---|
| Loan_ID | حذف | معرّف فريد يحتوي على 614 قيمة |
| Gender | استبعاد من التنبؤ | فرق الموافقة 2.16%، ومراعاة العدالة |
| Married | احتفاظ | فرق الموافقة 8.91% |
| Dependents | احتفاظ | فرق الموافقة 10.54% |
| Education | احتفاظ | فرق الموافقة 9.64% |
| Self_Employed | حذف | فرق الموافقة 0.50% فقط |
| ApplicantIncome | حذف | ارتباطه بـ Total_Income قوي ويساوي 0.893 |
| CoapplicantIncome | حذف | ارتباطه بالهدف ضعيف ويساوي -0.059 |
| LoanAmount | حذف | ارتباطه بالهدف ضعيف ويساوي -0.033 |
| Loan_Amount_Term | حذف | ارتباطه بالهدف ضعيف ويساوي -0.023 |
| Credit_History | احتفاظ | ارتباطه بالهدف 0.280 وفرق الموافقة 71.71% |
| Property_Area | احتفاظ | فرق الموافقة 15.37% |
| Total_Income | احتفاظ | يجمع دخل مقدم الطلب والشريك |
| Has_Coapplicant | احتفاظ | فرق الموافقة 7.01% |
| Loan_to_Income | حذف | ارتباطه بالهدف -0.086 |
| Monthly_Payment | حذف | ارتباطه بالهدف -0.012 |
| Payment_to_Income | احتفاظ | الفئات المرتبطة به أظهرت فرقًا 9.57% |
| Payment_Group | احتفاظ | يساعد على مقارنة نسب الموافقة بين فئات القسط |
| Income_Group | حذف | فرق الموافقة 1.10% فقط |
| Loan_Status | Target | هو العمود المطلوب التنبؤ به |

In [18]:
selected_features = [
    "Credit_History",
    "Property_Area",
    "Dependents",
    "Education",
    "Payment_Group",
    "Payment_to_Income",
    "Married",
    "Has_Coapplicant",
    "Total_Income",
    "Loan_Status"
]

model_df = df[selected_features]

model_df.to_csv("model_ready.csv", index=False)

print("File saved successfully: model_ready.csv")
print("Shape:", model_df.shape)
print("\nSelected columns:")
print(model_df.columns.tolist())

File saved successfully: model_ready.csv
Shape: (614, 10)

Selected columns:
['Credit_History', 'Property_Area', 'Dependents', 'Education', 'Payment_Group', 'Payment_to_Income', 'Married', 'Has_Coapplicant', 'Total_Income', 'Loan_Status']


 5. التقرير النهائي – Week 2 Final Report

  ملخص العمل
تم تحليل بيانات طلبات القروض البنكية باستخدام Pandas، وبدأت بفهم البيانات واستكشافها، ثم معالجة القيم الناقصة، وإنشاء خصائص جديدة، واختيار الخصائص المناسبة للتنبؤ بقرار الموافقة على القرض.

  أهم النتائج الرقمية
1. حجم البيانات الأصلي هو 614 صفًا و13 عمودًا.
2. كان مجموع القيم الناقصة 149 قيمة، وبعد التنظيف أصبح صفرًا.
3. لم توجد أي صفوف مكررة.
4. بلغت نسبة الموافقة عند وجود تاريخ ائتماني جيد (Credit_History = 1) حوالي 79.58%.
5. بلغت نسبة الموافقة عند وجود تاريخ ائتماني غير جيد (Credit_History = 0) حوالي 7.87%.
6. كان الفرق في نسب الموافقة حسب التاريخ الائتماني 71.71%.
7. حققت منطقة Semiurban نسبة موافقة 76.82%، وهي الأعلى بين المناطق.
8. بلغ الارتباط بين ApplicantIncome وTotal_Income حوالي 0.893، مما يدل على ارتباط قوي بينهما.
9. بلغ ارتباط Credit_History مع Loan_Status حوالي 0.280، وهو الأعلى بين الخصائص الرقمية.

 مقارنة فرضيات الأربعاء بالنتائج
- توقعت أن يكون Credit_History من أكثر العوامل تأثيرًا في قرار الموافقة، وأكدت النتائج صحة هذا التوقع.
- من النتائج التي لفتت انتباهي أن ارتباط الدخل بقرار الموافقة كان ضعيفًا، رغم أهمية الدخل في تقييم الوضع المالي.
- كان تأثير Gender ضعيفًا، حيث بلغ فرق الموافقة بين الفئات حوالي 2.16%.
- كان تأثير Self_Employed ضعيفًا أيضًا، بفرق بلغ حوالي 0.50%.
- أظهرت النتائج أن Property_Area له تأثير ملحوظ، بفرق موافقة بلغ 15.37%.

 توصية للبنك
أنصح البنك بإعطاء أهمية أكبر للتاريخ الائتماني عند تقييم طلبات القروض، مع دراسة الدخل وموقع العقار وبقية العوامل معًا، وعدم الاعتماد على عامل واحد فقط.

رأيي في استخدام Gender
أفضل عدم استخدام Gender كميزة للتنبؤ بقرار الموافقة؛ لأن الفرق بين الفئات ضعيف، وللمحافظة على العدالة وتجنب التمييز. ويمكن الاحتفاظ به بشكل منفصل لمراجعة عدالة قرارات النموذج.

# الخلاصة
تعلمت من هذا العمل أن فهم البيانات وتنظيفها وهندسة الخصائص واختيارها خطوات أساسية قبل بناء نموذج الذكاء الاصطناعي. كما أن قوة العلاقة بين الخصائص والهدف يجب أن تُفهم من خلال الأرقام، وليس التوقعات فقط.